<a href="https://colab.research.google.com/github/Mohsin-22/Assignment7-100_Gen-Ai_cohort/blob/main/Welcome_To_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 7: Sequential Agentic Workflows using LangGraph


In [1]:
!pip install langgraph langchain google-generativeai -q

In [16]:
!pip install langgraph -q

In [14]:

!pip uninstall google-generativeai -y
!pip install google-genai --upgrade


Found existing installation: google-generativeai 0.8.6
Uninstalling google-generativeai-0.8.6:
  Successfully uninstalled google-generativeai-0.8.6


In [18]:
class SIMState(TypedDict):
    aadhaar_valid: bool
    pan_valid: bool
    location: str
    previous_requests: int

    kyc_status: str
    fraud_risk: str
    final_decision: str

In [19]:
def kyc_verification(state: SIMState):
    if not state["aadhaar_valid"] or not state["pan_valid"]:
        state["kyc_status"] = "INVALID_DOCUMENT"
    else:
        state["kyc_status"] = "VERIFIED"
    return state

In [20]:
def fraud_detection(state: SIMState):

    if state["kyc_status"] == "INVALID_DOCUMENT":
        state["fraud_risk"] = "HIGH_RISK"

    elif state["previous_requests"] > 3:
        state["fraud_risk"] = "HIGH_RISK"

    elif "high fraud" in state["location"].lower():
        state["fraud_risk"] = "MEDIUM_RISK"

    else:
        state["fraud_risk"] = "LOW_RISK"

    return state

In [21]:
def final_decision(state: SIMState):

    if state["fraud_risk"] == "LOW_RISK":
        state["final_decision"] = "SIM Activated"

    elif state["fraud_risk"] == "MEDIUM_RISK":
        state["final_decision"] = "Manual Review"

    else:
        state["final_decision"] = "Rejected"

    return state

In [22]:
sim_graph = StateGraph(SIMState)

sim_graph.add_node("KYC", kyc_verification)
sim_graph.add_node("Fraud", fraud_detection)
sim_graph.add_node("Decision", final_decision)

sim_graph.set_entry_point("KYC")

sim_graph.add_edge("KYC", "Fraud")
sim_graph.add_edge("Fraud", "Decision")

sim_app = sim_graph.compile()

In [23]:
input_data = {
    "aadhaar_valid": True,
    "pan_valid": True,
    "location": "High fraud region",
    "previous_requests": 5,
}

result1 = sim_app.invoke(input_data)
print(result1)

{'aadhaar_valid': True, 'pan_valid': True, 'location': 'High fraud region', 'previous_requests': 5, 'kyc_status': 'VERIFIED', 'fraud_risk': 'HIGH_RISK', 'final_decision': 'Rejected'}


#
USE CASE 2: Healthcare Appointment Workflow

In [24]:
class HealthState(TypedDict):
    fever: float
    oxygen: int
    heart_rate: int
    duration_days: int
    age: int
    conditions: str

    severity: str
    priority: str
    consultation: str


In [25]:
def symptom_checker(state: HealthState):

    if state["oxygen"] < 90 or state["heart_rate"] > 120:
        state["severity"] = "CRITICAL"

    elif state["fever"] > 101:
        state["severity"] = "MODERATE"

    else:
        state["severity"] = "STABLE"

    return state


In [26]:
def medical_prioritization(state: HealthState):

    if state["severity"] == "CRITICAL":
        state["priority"] = "EMERGENCY"

    elif state["severity"] == "MODERATE" and state["age"] > 60:
        state["priority"] = "PRIORITY_CONSULTATION"

    elif "diabetes" in state["conditions"].lower():
        state["priority"] = "PRIORITY_CONSULTATION"

    else:
        state["priority"] = "REGULAR_CONSULTATION"

    return state

In [27]:
def assign_consultation(state: HealthState):

    if state["priority"] == "EMERGENCY":
        state["consultation"] = "ICU/ER"

    elif state["priority"] == "PRIORITY_CONSULTATION":
        state["consultation"] = "Specialist Doctor"

    else:
        state["consultation"] = "General Physician"

    return state

In [28]:
health_graph = StateGraph(HealthState)

health_graph.add_node("Symptoms", symptom_checker)
health_graph.add_node("Priority", medical_prioritization)
health_graph.add_node("Consult", assign_consultation)

health_graph.set_entry_point("Symptoms")

health_graph.add_edge("Symptoms", "Priority")
health_graph.add_edge("Priority", "Consult")

health_app = health_graph.compile()


In [29]:
patient = {
    "fever": 102,
    "oxygen": 95,
    "heart_rate": 110,
    "duration_days": 3,
    "age": 65,
    "conditions": "Diabetes"
}

result2 = health_app.invoke(patient)
print(result2)

{'fever': 102, 'oxygen': 95, 'heart_rate': 110, 'duration_days': 3, 'age': 65, 'conditions': 'Diabetes', 'severity': 'MODERATE', 'priority': 'PRIORITY_CONSULTATION', 'consultation': 'Specialist Doctor'}
